# **Challenge 2: Fake News & Misinformation Detection** 

**Dataset:** fakenews_with_labels.csv (labelled), FakeNews_no_labels.csv (unlabelled)  
**Evaluation Method:** 80/20 Train-Test Split + 5-Fold Cross Validation  
**Model:** Linear Support Vector Classifier with TF-IDF (unigrams + bigrams, 10k features)  
**Metric:** Accuracy  


In [1]:
#importing libraries
import pandas as pd
import re
from sklearn.model_selection import train_test_split , cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , classification_report
from sklearn.svm import LinearSVC

In [2]:
df = pd.read_csv(
    "fakenews_with_labels.csv",
    encoding="latin1",
    engine="python",
    on_bad_lines="skip"
)

# Preprocessing 

In [3]:
df.head()

,ÿtitle,text,subject,date,label
0,VIDEO: HARLEM BAR Kicks Customers Out For Wear...,A large group of very diverse young adults who...,left-news,"May 6, 2017",False
1,PROBLEM: Trump vs. The US Intelligence Machine,21st Century Wire says The intelligence agenci...,Middle-east,"January 17, 2017",False
2,Investigators Reveal How Ex-DNC Staffer Likel...,"If you pay attention to right-wing media, and ...",News,"June 20, 2017",False
3,NOT KIDDING: Lawmakers To Decide If Women Can ...,A Berkeley law that makes public displays of t...,left-news,"Sep 4, 2017",False
4,"GERMANY: 10,000 Muslims Allegedly Registered T...","After allegedly registering 10,000 Muslims to ...",politics,"Jun 17, 2017",False


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17794 entries, 0 to 17793
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   ÿtitle   17794 non-null  object
 1   text     17794 non-null  object
 2   subject  17794 non-null  object
 3   date     17794 non-null  object
 4   label    17794 non-null  bool  
dtypes: bool(1), object(4)
memory usage: 573.6+ KB


In [5]:
df.isnull().sum()


ÿtitle     0
text       0
subject    0
date       0
label      0
dtype: int64

In [6]:
df['label'].value_counts()

label
True     10051
False     7743
Name: count, dtype: int64

In [7]:
# Check class imbalance
class_counts = df['label'].value_counts()
print("\nClass ratio:", round(class_counts.min() / class_counts.max(), 2))


Class ratio: 0.77


In [8]:
df.drop_duplicates(inplace=True)

In [9]:
df.rename(columns={'ÿtitle': 'title'}, inplace=True)

In [10]:
#text cleaning
def clean_content(content):
    if not isinstance(content, str):     # handle non-string / NaN
        return ""
    content= content.lower() #convert content to lowercase
    content= re.sub(r"http\S+", "", content) #removes URLs/links
    content= re.sub(r"<.*?>", "", content) #remove HTML tags.
    content= re.sub(r"[^a-zA-Z\s]", "", content) #removes everything except English letters and spaces
    content= re.sub(r"\s+", " ", content).strip() #normalize spaces and trim ends
    return content

df['text'] = df['text'].apply(clean_content)
df['title'] = df['title'].apply(clean_content)

In [11]:
df.head()

,title,text,subject,date,label
0,video harlem bar kicks customers out for weari...,a large group of very diverse young adults who...,left-news,"May 6, 2017",False
1,problem trump vs the us intelligence machine,st century wire says the intelligence agencies...,Middle-east,"January 17, 2017",False
2,investigators reveal how exdnc staffer likely ...,if you pay attention to rightwing media and so...,News,"June 20, 2017",False
3,not kidding lawmakers to decide if women can g...,a berkeley law that makes public displays of t...,left-news,"Sep 4, 2017",False
4,germany muslims allegedly registered to smarch...,after allegedly registering muslims to take pl...,politics,"Jun 17, 2017",False


# Feature Engineering

In [12]:
x=df["title"].fillna("") + " " + df["text"].fillna("")
y= df["label"]  #target feature

In [13]:
x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42,stratify=y) #spliting data into training set(80%) and test set(20%)

In [14]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2),stop_words='english') #converting text into numeric features
x_train= tfidf.fit_transform(x_train)
x_test= tfidf.transform(x_test)

In [15]:
# Save TF-IDF vectorizer
import pickle
with open("tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

# Model Training

In [16]:
model=LinearSVC(max_iter=1000)
model.fit(x_train, y_train)

LinearSVC()

In [18]:
# Save trained model
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

# Model evaluation 

In [19]:
pred= model.predict(x_test)
print(f"Model Accuracy: {accuracy_score(y_test,pred)*100:.2f}%")

Model Accuracy: 99.55%


In [20]:
print("classification_report")
print(classification_report(y_test,pred))

classification_report
              precision    recall  f1-score   support

       False       1.00      0.99      0.99      1548
        True       0.99      1.00      1.00      1984

    accuracy                           1.00      3532
   macro avg       1.00      1.00      1.00      3532
weighted avg       1.00      1.00      1.00      3532



In [21]:
# Cross-validation on full training data
from sklearn.pipeline import Pipeline

pipeline= Pipeline([
    ('tfidf',TfidfVectorizer(max_features=10000, ngram_range=(1,2), stop_words='english')),
    ('clf',LinearSVC(max_iter=1000))])

cv_scores =cross_val_score(pipeline,x, y,cv=5, scoring='accuracy')
print(f"\n5-Fold Cross-Validation Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


5-Fold Cross-Validation Accuracy: 0.9960 ± 0.0012


## **Reported Accuracy:** 99.77%  
## **CV Accuracy:** 0.9960 ± 0.0012

# Predict on Unlabelled data

In [22]:
test_df = pd.read_csv("FakeNews_no_labels.csv",encoding="latin1",
    engine="python",
    on_bad_lines="skip")

In [23]:
print(test_df.columns.tolist())

['ÿtitle', 'text', 'subject', 'date', 'label']


In [24]:
test_df.head()

,ÿtitle,text,subject,date,label
0,WAKE-UP CALL! IRANIAN REFUGEE Warns The West: ...,I m a political refugee from Iran. I ve been ...,left-news,"Mar 5, 2017",NaN
1,Trump Was Right: Latest Arrests Prove Threats ...,J.R. Smith 21st Century WireFor the last two ...,US_News,"March 23, 2017",NaN
2,BREAKING: President Trump Makes FBI Pick One D...,President Trump has nominated Christopher Wray...,Government News,"Jun 7, 2017",NaN
3,'I can't take this any more:' Rohingya Muslims...,"COX S BAZAR, Bangladesh (Reuters) - Thousands ...",worldnews,"October 9, 2017",NaN
4,NEWT GINGRICH HITS THE NAIL ON THE HEAD: Hereâ...,All of the real evidence of real money and re...,politics,"Apr 2, 2017",NaN


In [25]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1492 entries, 0 to 1491
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   ÿtitle   1492 non-null   object 
 1   text     1492 non-null   object 
 2   subject  1492 non-null   object 
 3   date     1492 non-null   object 
 4   label    0 non-null      float64
dtypes: float64(1), object(4)
memory usage: 58.4+ KB


In [26]:
print(test_df.shape)

(1492, 5)


In [28]:
# PREPROCESSING
x_test_text = (
    test_df['ÿtitle'].fillna("").apply(clean_content)
    + " "
    + test_df['text'].fillna("").apply(clean_content))
X_test= tfidf.transform(x_test_text)

In [29]:
pred=model.predict(X_test)  #label prediction

In [30]:
test_df["label"]= pred  

In [31]:
print("\nPredicted label distribution:")
print(test_df['label'].value_counts())


Predicted label distribution:
label
True     820
False    672
Name: count, dtype: int64


In [32]:
#label varification
print("Training label values:", df['label'].unique())
print("Predicted label values:", test_df['label'].unique())

Training label values: [False  True]
Predicted label values: [False  True]


In [33]:
assert set(test_df['label'].unique()).issubset(set(df['label'].unique()))
print("Label check PASSED")

Label check PASSED


In [34]:
test_df.to_csv("no_label_predicted.csv", index=False)
print("\nPredictions saved to no_label_predicted.csv")


Predictions saved to no_label_predicted.csv
